# Customer Behaviour & Personalisation in Fashion Retail

## Complete Dissertation Data Analysis

This notebook analyses the primary survey data collected for the dissertation **Customer Behaviour & Personalisation in Fashion Retail**.

The workflow follows the methodology structure of the supplied reference notebook: data audit, cleaning, descriptive analysis, correlation, simple linear regression, regression assumptions, multiple linear regression, model comparison and business interpretation.

**Important:** The survey is cross-sectional. Therefore, the results identify statistical associations and should not be presented as proof of causation.

## 1. Methodology used in this notebook

The analysis includes:

1. Data loading and initial audit
2. Consent-based data cleaning
3. Missing-value assessment
4. Duplicate checking
5. Conversion of ordinal income, spending and purchase-frequency bands
6. Likert-scale coding from 1 to 5
7. Personalisation composite score
8. Brand-reputation composite score
9. Descriptive statistics
10. Interactive Plotly Express visualisations
11. Animated visualisations by age group
12. Pearson correlation
13. Correlation heatmap
14. Simple linear regression
15. Linearity, homoscedasticity and residual diagnostics
16. Shapiro-Wilk normality test
17. Durbin-Watson test
18. Multiple linear regression
19. VIF multicollinearity test
20. R², adjusted R² and AIC model comparison
21. Hypothesis testing
22. Dissertation-ready business interpretation

In [95]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', None)
print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load the dataset

The notebook searches for the Excel survey file automatically, avoiding machine-specific file paths.

In [96]:
df = pd.read_csv("Customer Behaviour & Personalisation in Fashion Retail (Responses) - Form responses 1.csv")
# Display first 5 rows
df.head()

,Timestamp,Do you consent to participate in this research study?,What is your age?,What is your gender?,What is your monthly disposable income?,Approximately how much do you spend on fashion products each month?,How many fashion purchases do you typically make in a month?,Which fashion product category do you purchase most frequently?,Which shopping channel do you use most frequently?,Which fashion brand do you purchase from most frequently?,What is the single most important factor influencing your fashion purchases?,To what extent do discounts influence your fashion purchasing decisions?,To what extent do social media platforms or influencers influence your fashion purchasing decisions?,To what extent do you agree with the following statements? [I regularly receive personalised recommendations from fashion retailers.],To what extent do you agree with the following statements? [Personalised recommendations improve my shopping experience.],To what extent do you agree with the following statements? [I am more likely to purchase from a retailer that offers personalised recommendations.],To what extent do you agree with the following statements? [I feel valued when a retailer understands my shopping preferences.],To what extent do you agree with the following statements? [Personalised communication increases my loyalty to a fashion retailer.],Please indicate your level of agreement with the following statements. [I trust reputable fashion brands.],Please indicate your level of agreement with the following statements. [A brand's reputation influences my purchase decisions.],Please indicate your level of agreement with the following statements. [I would recommend a fashion brand with a good reputation.],Please indicate your level of agreement with the following statements. [I am loyal to brands that meet my expectations.],Please indicate your level of agreement with the following statements. [A positive shopping experience improves my perception of a brand.],"In your opinion, what could fashion retailers do to improve your shopping experience through personalisation?",Is there anything else you would like to share about your fashion shopping behaviour?,Email address
0,21/07/2026 20:09:20,I agree to participate in this research study.,18–24,Male,"£1,000–£1,999",£50–£99,1–2,Accessories,Both equally,Mango,Comfort,3.0,3.0,Agree,Agree,Agree,Agree,Agree,Neutral,Neutral,Neutral,Neutral,Neutral,NaN,NaN,NaN
1,21/07/2026 20:11:03,I agree to participate in this research study.,18–24,Female,"£2,000–£2,999",£200–£299,3–4,Clothing,Online,zara,Comfort,2.0,2.0,Disagree,Agree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,To improve more efficiency and engagement in c...,No,NaN
2,22/07/2026 19:24:23,I agree to participate in this research study.,35–44,Female,"Less than £1,000",Less than £50,3–4,Clothing,Both equally,"Mango, Zara, H&M",Comfort,3.0,3.0,Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Strongly Agree,Agree,Agree,Strongly Agree,Agree,Quality,No,NaN
3,22/07/2026 19:25:35,I agree to participate in this research study.,18–24,Male,"Less than £1,000",£50–£99,1–2,Clothing,Physical stores,Uniqlo,Fashion trends,4.0,5.0,Neutral,Agree,Neutral,Neutral,Agree,Agree,Agree,Agree,Agree,Agree,NaN,NaN,NaN
4,22/07/2026 19:29:50,I agree to participate in this research study.,25–34,Male,"£2,000–£2,999",£50–£99,3–4,"Clothing, Footwear",Physical stores,"Mango, Zara, H&M",Product quality,4.0,1.0,Agree,Agree,Agree,Agree,Strongly Agree,Strongly Disagree,Strongly Disagree,Strongly Disagree,Strongly Agree,Strongly Agree,NaN,NaN,NaN


## 3. Initial data audit

This section checks the size, variables, data types and duplicate observations before analysis.

In [97]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nDuplicate rows:", df.duplicated().sum())

Dataset shape: (70, 26)

Column names:
1. Timestamp
2. Do you consent to participate in this research study?
3. What is your age?
4. What is your gender?
5. What is your monthly disposable income?
6. Approximately how much do you spend on fashion products each month?
7. How many fashion purchases do you typically make in a month?
8. Which fashion product category do you purchase most frequently?
9. Which shopping channel do you use most frequently?
10. Which fashion brand do you purchase from most frequently?
11. What is the single most important factor influencing your fashion purchases?
12. To what extent do discounts influence your fashion purchasing decisions?
13. To what extent do social media platforms or influencers influence your fashion purchasing decisions?
14. To what extent do you agree with the following statements? [I regularly receive personalised recommendations from fashion retailers.]
15. To what extent do you agree with the following statements? [Personalised recomme

,Data Type
Timestamp,object
Do you consent to participate in this research study?,object
What is your age?,object
What is your gender?,object
What is your monthly disposable income?,object
Approximately how much do you spend on fashion products each month?,object
How many fashion purchases do you typically make in a month?,object
Which fashion product category do you purchase most frequently?,object
Which shopping channel do you use most frequently?,object
Which fashion brand do you purchase from most frequently?,object



Duplicate rows: 0


## 4. Missing values

The raw dataset contains one non-consenting blank response. Because participation consent is a research requirement, this response is excluded rather than imputed.

The two open-ended questions are optional qualitative questions, so missing answers are labelled **No response**.

In [98]:
missing_report = pd.DataFrame({
    'Missing Values': df.isna().sum(),
    'Percentage': (df.isna().sum() / len(df) * 100).round(2)
}).sort_values('Percentage', ascending=False)

display(missing_report[missing_report['Missing Values'] > 0])

,Missing Values,Percentage
Email address,70,100.00
Is there anything else you would like to share about your fashion shopping behaviour?,44,62.86
"In your opinion, what could fashion retailers do to improve your shopping experience through personalisation?",38,54.29
To what extent do you agree with the following statements? [Personalised recommendations improve my shopping experience.],1,1.43
Please indicate your level of agreement with the following statements. [A positive shopping experience improves my perception of a brand.],1,1.43
Please indicate your level of agreement with the following statements. [I am loyal to brands that meet my expectations.],1,1.43
Please indicate your level of agreement with the following statements. [I would recommend a fashion brand with a good reputation.],1,1.43
Please indicate your level of agreement with the following statements. [A brand's reputation influences my purchase decisions.],1,1.43
Please indicate your level of agreement with the following statements. [I trust reputable fashion brands.],1,1.43
To what extent do you agree with the following statements? [Personalised communication increases my loyalty to a fashion retailer.],1,1.43


## 5. Clean the dataset

Only respondents who agreed to participate are retained. Administrative fields such as timestamp, consent and email are removed from the analytical dataset.

No artificial zero values are inserted.

In [99]:
consent_col = 'Do you consent to participate in this research study?'

df_clean = df[
    df[consent_col] == 'I agree to participate in this research study.'
].copy()

open_text = [
    'In your opinion, what could fashion retailers do to improve your shopping experience through personalisation?',
    'Is there anything else you would like to share about your fashion shopping behaviour?'
]

for col in open_text:
    df_clean[col] = df_clean[col].fillna('No response')

df_clean = df_clean.drop(
    columns=['Timestamp', consent_col, 'Email address'],
    errors='ignore'
)

df_clean = df_clean.drop_duplicates().reset_index(drop=True)

print("Original respondents:", len(df))
print("Consenting respondents:", len(df_clean))
print("Remaining missing cells:", df_clean.isna().sum().sum())
print("Duplicate rows after cleaning:", df_clean.duplicated().sum())

Original respondents: 70
Consenting respondents: 68
Remaining missing cells: 0
Duplicate rows after cleaning: 0


## 6. Convert ordinal survey ranges

Income, monthly fashion spending and purchase frequency were collected as categories. Approximate midpoints are used to create numerical variables suitable for correlation and regression.

These are analytical approximations and should be acknowledged as a limitation.

In [100]:
income_map = {
    'Less than £1,000': 500,
    '£1,000–£1,999': 1500,
    '£2,000–£2,999': 2500,
    '£3,000–£3,999': 3500,
    '£4,000–£4,999': 4500,
    '£5,000 or above': 5500
}

spending_map = {
    'Less than £50': 25,
    '£50–£99': 75,
    '£100–£199': 150,
    '£200–£299': 250,
    '£300–£499': 400,
    '£500 or above': 500
}

purchase_map = {
    '1–2': 1.5,
    '3–4': 3.5,
    '5–6': 5.5,
    '7 or more': 7
}

df_clean['Income_Numeric'] = df_clean[
    'What is your monthly disposable income?'
].map(income_map)

df_clean['Fashion_Spending'] = df_clean[
    'Approximately how much do you spend on fashion products each month?'
].map(spending_map)

df_clean['Purchase_Frequency'] = df_clean[
    'How many fashion purchases do you typically make in a month?'
].map(purchase_map)

print("Unmapped income:", df_clean['Income_Numeric'].isna().sum())
print("Unmapped spending:", df_clean['Fashion_Spending'].isna().sum())
print("Unmapped purchase frequency:", df_clean['Purchase_Frequency'].isna().sum())

Unmapped income: 0
Unmapped spending: 0
Unmapped purchase frequency: 0


## 7. Likert-scale coding and composite variables

The five personalisation questions and five brand-reputation questions use a five-point Likert scale.

- Strongly Disagree = 1
- Disagree = 2
- Neutral = 3
- Agree = 4
- Strongly Agree = 5

The mean of the five items creates each composite score.

In [101]:
likert_map = {
    'Strongly Disagree': 1,
    'Disagree': 2,
    'Neutral': 3,
    'Agree': 4,
    'Strongly Agree': 5
}

personalisation_cols = [
    'To what extent do you agree with the following statements? [I regularly receive personalised recommendations from fashion retailers.]',
    'To what extent do you agree with the following statements? [Personalised recommendations improve my shopping experience.]',
    'To what extent do you agree with the following statements? [I am more likely to purchase from a retailer that offers personalised recommendations.]',
    'To what extent do you agree with the following statements? [I feel valued when a retailer understands my shopping preferences.]',
    'To what extent do you agree with the following statements? [Personalised communication increases my loyalty to a fashion retailer.]'
]

brand_cols = [
    "Please indicate your level of agreement with the following statements. [I trust reputable fashion brands.]",
    "Please indicate your level of agreement with the following statements. [A brand's reputation influences my purchase decisions.]",
    "Please indicate your level of agreement with the following statements. [I would recommend a fashion brand with a good reputation.]",
    "Please indicate your level of agreement with the following statements. [I am loyal to brands that meet my expectations.]",
    "Please indicate your level of agreement with the following statements. [A positive shopping experience improves my perception of a brand.]"
]

for col in personalisation_cols + brand_cols:
    df_clean[col + '_Score'] = df_clean[col].map(likert_map)

df_clean['Personalisation_Score'] = df_clean[
    [c + '_Score' for c in personalisation_cols]
].mean(axis=1)

df_clean['Brand_Reputation_Score'] = df_clean[
    [c + '_Score' for c in brand_cols]
].mean(axis=1)

df_clean['Discount_Influence'] = pd.to_numeric(
    df_clean['To what extent do discounts influence your fashion purchasing decisions?'],
    errors='coerce'
)

df_clean['Social_Media_Influence'] = pd.to_numeric(
    df_clean['To what extent do social media platforms or influencers influence your fashion purchasing decisions?'],
    errors='coerce'
)

analysis_cols = [
    'Income_Numeric',
    'Fashion_Spending',
    'Purchase_Frequency',
    'Discount_Influence',
    'Social_Media_Influence',
    'Personalisation_Score',
    'Brand_Reputation_Score'
]

display(df_clean[analysis_cols].isna().sum().to_frame('Missing values'))

,Missing values
Income_Numeric,0
Fashion_Spending,0
Purchase_Frequency,0
Discount_Influence,0
Social_Media_Influence,0
Personalisation_Score,0
Brand_Reputation_Score,0


## 8. Final analytical dataset

Short variable names are used for the statistical models to avoid errors caused by long questionnaire column names.

In [102]:
analysis_df = df_clean[analysis_cols].copy()

analysis_df['Age_Group'] = df_clean['What is your age?'].values
analysis_df['Gender'] = df_clean['What is your gender?'].values
analysis_df['Brand'] = df_clean[
    'Which fashion brand do you purchase from most frequently?'
].values
analysis_df['Purchase_Factor'] = df_clean[
    'What is the single most important factor influencing your fashion purchases?'
].values

# Complete-case check for the analytical variables
analysis_df = analysis_df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

print("Final analytical sample:", len(analysis_df))
display(analysis_df.head())

Final analytical sample: 68


,Income_Numeric,Fashion_Spending,Purchase_Frequency,Discount_Influence,Social_Media_Influence,Personalisation_Score,Brand_Reputation_Score,Age_Group,Gender,Brand,Purchase_Factor
0,1500,75,1.5,3.0,3.0,4.0,3.0,18–24,Male,Mango,Comfort
1,2500,250,3.5,2.0,2.0,4.0,5.0,18–24,Female,zara,Comfort
2,500,25,3.5,3.0,3.0,4.8,4.4,35–44,Female,"Mango, Zara, H&M",Comfort
3,500,75,1.5,4.0,5.0,3.4,4.0,18–24,Male,Uniqlo,Fashion trends
4,2500,75,3.5,4.0,1.0,4.2,2.6,25–34,Male,"Mango, Zara, H&M",Product quality


# Part A — Interactive Plotly visualisations

Plotly Express is used instead of Seaborn for the main visualisations. Animation is only used across meaningful age groups because this is a cross-sectional survey and does not contain a time series.

## 9. Age distribution by gender

In [103]:
fig = px.histogram(
    analysis_df,
    x='Age_Group',
    color='Gender',
    barmode='group',
    title='Age Distribution of Respondents by Gender',
    labels={'Age_Group':'Age Group','count':'Number of Respondents'}
)
fig.show()

## 10. Monthly fashion spending by gender

In [104]:
fig = px.box(
    analysis_df,
    x='Gender',
    y='Fashion_Spending',
    points='all',
    title='Monthly Fashion Spending by Gender',
    labels={
        'Gender':'Gender',
        'Fashion_Spending':'Estimated Monthly Fashion Spending (£)'
    }
)
fig.show()

## 11. Disposable income distribution

In [105]:
fig = px.histogram(
    analysis_df,
    x='Income_Numeric',
    color='Age_Group',
    nbins=6,
    title='Monthly Disposable Income Distribution by Age Group',
    labels={
        'Income_Numeric':'Approximate Monthly Disposable Income (£)',
        'count':'Respondents'
    }
)
fig.show()

## 12. Income versus fashion spending

The OLS trendline provides a visual representation of the relationship later tested formally using Pearson correlation and simple linear regression.

In [106]:
fig = px.scatter(
    analysis_df,
    x='Income_Numeric',
    y='Fashion_Spending',
    color='Gender',
    size='Purchase_Frequency',
    hover_data=['Age_Group','Personalisation_Score','Brand_Reputation_Score'],
    trendline='ols',
    title='Disposable Income vs Monthly Fashion Spending',
    labels={
        'Income_Numeric':'Approximate Monthly Disposable Income (£)',
        'Fashion_Spending':'Approximate Monthly Fashion Spending (£)',
        'Purchase_Frequency':'Purchase Frequency'
    },
    size_max=40
)
fig.show()

## 13. Personalisation versus fashion spending

In [107]:
fig = px.scatter(
    analysis_df,
    x='Personalisation_Score',
    y='Fashion_Spending',
    color='Gender',
    size='Purchase_Frequency',
    hover_data=['Age_Group','Income_Numeric','Brand_Reputation_Score'],
    trendline='ols',
    title='Personalisation Experience vs Monthly Fashion Spending',
    labels={
        'Personalisation_Score':'Personalisation Score (1–5)',
        'Fashion_Spending':'Approximate Monthly Fashion Spending (£)',
        'Purchase_Frequency':'Purchase Frequency'
    },
    size_max=40
)
fig.show()

## 14. Animated income and spending by age group

This is the closest equivalent to the animated Plotly examples supplied. The animation compares age groups rather than pretending that the survey contains observations over time.

In [108]:
fig = px.scatter(
    analysis_df,
    x='Income_Numeric',
    y='Fashion_Spending',
    animation_frame='Age_Group',
    color='Gender',
    size='Purchase_Frequency',
    hover_data=['Personalisation_Score','Brand_Reputation_Score'],
    title='Animated Income vs Fashion Spending Across Age Groups',
    labels={
        'Income_Numeric':'Approximate Monthly Disposable Income (£)',
        'Fashion_Spending':'Approximate Monthly Fashion Spending (£)',
        'Purchase_Frequency':'Purchase Frequency'
    },
    size_max=45
)
fig.show()

## 15. Animated personalisation and spending by age group

In [109]:
fig = px.scatter(
    analysis_df,
    x='Personalisation_Score',
    y='Fashion_Spending',
    animation_frame='Age_Group',
    color='Gender',
    size='Purchase_Frequency',
    hover_data=['Income_Numeric','Brand_Reputation_Score'],
    title='Animated Personalisation Experience vs Fashion Spending by Age Group',
    labels={
        'Personalisation_Score':'Personalisation Score (1–5)',
        'Fashion_Spending':'Approximate Monthly Fashion Spending (£)',
        'Purchase_Frequency':'Purchase Frequency'
    },
    size_max=45
)
fig.show()

## 16. Most important purchase factors

In [110]:
factor_counts = (
    analysis_df['Purchase_Factor']
    .value_counts()
    .rename_axis('Purchase_Factor')
    .reset_index(name='Respondents')
)

fig = px.bar(
    factor_counts,
    x='Purchase_Factor',
    y='Respondents',
    text='Respondents',
    title='Most Important Factors Influencing Fashion Purchases',
    labels={'Purchase_Factor':'Purchase Factor','Respondents':'Respondents'}
)
fig.show()

## 17. Personalisation and brand-reputation scores

In [111]:
score_long = analysis_df[
    ['Personalisation_Score','Brand_Reputation_Score']
].melt(var_name='Construct', value_name='Score')

fig = px.box(
    score_long,
    x='Construct',
    y='Score',
    points='all',
    title='Personalisation and Brand Reputation Scores',
    labels={'Construct':'Construct','Score':'Mean Likert Score (1–5)'}
)
fig.show()

# Part B — Descriptive statistics and correlation

Descriptive statistics establish the central tendency and spread of the main analytical variables before inferential testing.

In [112]:
descriptive = analysis_df[analysis_cols].describe().T.round(3)
display(descriptive)

,count,mean,std,min,25%,50%,75%,max
Income_Numeric,68.0,1588.235,1047.175,500.0,500.0,1500.0,2500.0,5500.0
Fashion_Spending,68.0,126.103,99.854,25.0,75.0,75.0,150.0,500.0
Purchase_Frequency,68.0,2.890,1.401,1.5,1.5,3.5,3.5,7.0
Discount_Influence,68.0,3.456,1.177,1.0,3.0,3.5,4.0,5.0
Social_Media_Influence,68.0,3.118,1.179,1.0,2.0,3.0,4.0,5.0
Personalisation_Score,68.0,3.621,0.920,1.2,3.2,3.8,4.2,5.0
Brand_Reputation_Score,68.0,3.926,1.043,1.0,3.6,4.0,4.6,5.0


## 18. Correlation matrix and interactive heatmap

Pearson correlation is used for the numerical analytical variables. The heatmap provides a compact visual summary.

In [113]:
corr_matrix = analysis_df[analysis_cols].corr()

fig = px.imshow(
    corr_matrix,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='Viridis',
    title='Correlation Heatmap of Key Fashion-Retail Variables',
    labels={'color':'Correlation'}
)
fig.show()

## 19. Pearson correlation — income and fashion spending

**H1:** Disposable income is significantly associated with monthly fashion spending.

**H0:** Disposable income is not significantly associated with monthly fashion spending.

In [114]:
r_income, p_income = stats.pearsonr(
    analysis_df['Income_Numeric'],
    analysis_df['Fashion_Spending']
)

print(f"Correlation (r): {r_income:.3f}")
print(f"P-value: {p_income:.4f}")
print(f"N: {len(analysis_df)}")
print(
    "Decision:",
    "Reject H0 at 5% significance."
    if p_income < 0.05
    else "Fail to reject H0 at 5% significance."
)

Correlation (r): 0.285
P-value: 0.0187
N: 68
Decision: Reject H0 at 5% significance.


## 20. Pearson correlation — personalisation and fashion spending

**H2:** Personalisation experience is significantly associated with monthly fashion spending.

**H0:** Personalisation experience is not significantly associated with monthly fashion spending.

In [115]:
r_pers, p_pers = stats.pearsonr(
    analysis_df['Personalisation_Score'],
    analysis_df['Fashion_Spending']
)

print(f"Correlation (r): {r_pers:.3f}")
print(f"P-value: {p_pers:.4f}")
print(f"N: {len(analysis_df)}")
print(
    "Decision:",
    "Reject H0 at 5% significance."
    if p_pers < 0.05
    else "Fail to reject H0 at 5% significance."
)

Correlation (r): 0.016
P-value: 0.8969
N: 68
Decision: Fail to reject H0 at 5% significance.


# Part C — Simple linear regression

The reference methodology uses simple OLS regression before moving to multiple regression. Two simple models are used here: income and personalisation as predictors of monthly fashion spending.

## 21. Simple linear regression — Income

In [116]:
model1 = ols(
    'Fashion_Spending ~ Income_Numeric',
    data=analysis_df
).fit()

print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:       Fashion_Spending   R-squared:                       0.081
Model:                            OLS   Adj. R-squared:                  0.067
Method:                 Least Squares   F-statistic:                     5.814
Date:                Sun, 23 Aug 2026   Prob (F-statistic):             0.0187
Time:                        01:43:13   Log-Likelihood:                -406.17
No. Observations:                  68   AIC:                             816.3
Df Residuals:                      66   BIC:                             820.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         83.0114     21.358      3.

## 22. Simple regression visualisation — Income

In [117]:
fig = px.scatter(
    analysis_df,
    x='Income_Numeric',
    y='Fashion_Spending',
    color='Gender',
    trendline='ols',
    title='Simple Linear Regression: Income and Fashion Spending',
    labels={
        'Income_Numeric':'Approximate Monthly Disposable Income (£)',
        'Fashion_Spending':'Approximate Monthly Fashion Spending (£)'
    }
)
fig.show()

## 23. Simple linear regression — Personalisation

In [118]:
model2 = ols(
    'Fashion_Spending ~ Personalisation_Score',
    data=analysis_df
).fit()

print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:       Fashion_Spending   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.015
Method:                 Least Squares   F-statistic:                   0.01691
Date:                Sun, 23 Aug 2026   Prob (F-statistic):              0.897
Time:                        01:43:13   Log-Likelihood:                -409.03
No. Observations:                  68   AIC:                             822.1
Df Residuals:                      66   BIC:                             826.5
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept               119.81

## 24. Simple regression visualisation — Personalisation

In [119]:
fig = px.scatter(
    analysis_df,
    x='Personalisation_Score',
    y='Fashion_Spending',
    color='Gender',
    trendline='ols',
    title='Simple Linear Regression: Personalisation and Fashion Spending',
    labels={
        'Personalisation_Score':'Personalisation Score (1–5)',
        'Fashion_Spending':'Approximate Monthly Fashion Spending (£)'
    }
)
fig.show()

# Part D — OLS assumptions and diagnostics

The following tests and plots assess linearity, homoscedasticity, residual normality and independence. These diagnostics should be interpreted together.

## 25. Linearity check

In [120]:
fig = px.scatter(
    analysis_df,
    x='Income_Numeric',
    y='Fashion_Spending',
    trendline='ols',
    title='Linearity Check: Income vs Fashion Spending',
    labels={
        'Income_Numeric':'Income (£)',
        'Fashion_Spending':'Fashion Spending (£)'
    }
)
fig.show()

## 26. Homoscedasticity — residuals versus fitted values

A roughly random spread around zero without a strong funnel pattern supports constant residual variance.

In [121]:
resid1 = pd.DataFrame({
    'Fitted': model1.fittedvalues,
    'Residual': model1.resid
})

fig = px.scatter(
    resid1,
    x='Fitted',
    y='Residual',
    trendline='lowess',
    title='Homoscedasticity Check: Residuals vs Fitted Values',
    labels={'Fitted':'Fitted Values','Residual':'Residual'}
)
fig.add_hline(y=0, line_dash='dash')
fig.show()

## 27. Residual histogram

In [122]:
fig = px.histogram(
    x=model1.resid,
    nbins=12,
    marginal='box',
    title='Histogram of Simple Regression Residuals',
    labels={'x':'Residual'}
)
fig.show()

## 28. Q-Q plot

The Q-Q plot compares observed residual quantiles with theoretical normal quantiles.

In [123]:
theoretical, ordered = stats.probplot(
    model1.resid, dist='norm'
)[0]

slope, intercept = stats.probplot(
    model1.resid, dist='norm'
)[1][:2]

qq = pd.DataFrame({
    'Theoretical': theoretical,
    'Observed': ordered
})

xline = np.array([
    qq['Theoretical'].min(),
    qq['Theoretical'].max()
])
yline = intercept + slope * xline

fig = px.scatter(
    qq,
    x='Theoretical',
    y='Observed',
    title='Q-Q Plot of Regression Residuals',
    labels={
        'Theoretical':'Theoretical Quantiles',
        'Observed':'Observed Residual Quantiles'
    }
)

fig.add_trace(
    go.Scatter(
        x=xline,
        y=yline,
        mode='lines',
        name='Reference line'
    )
)

fig.show()

## 29. Shapiro-Wilk normality test

A p-value above 0.05 indicates insufficient evidence of non-normality at the 5% level.

In [124]:
shapiro1 = stats.shapiro(model1.resid)

print(f"Statistic: {shapiro1.statistic:.4f}")
print(f"P-value: {shapiro1.pvalue:.4f}")

if shapiro1.pvalue >= 0.05:
    print("Conclusion: No statistically significant departure from normality at 5%.")
else:
    print("Conclusion: Evidence of non-normal residuals at 5%; inspect the Q-Q plot.")

Statistic: 0.8894
P-value: 0.0000
Conclusion: Evidence of non-normal residuals at 5%; inspect the Q-Q plot.


## 30. Durbin-Watson test

A statistic close to 2 suggests little residual autocorrelation.

In [125]:
dw1 = durbin_watson(model1.resid)
print(f"Durbin-Watson statistic: {dw1:.3f}")

Durbin-Watson statistic: 2.593


# Part E — Multiple linear regression

The multiple-regression models are built progressively, similar to the staged modelling approach in the supplied reference notebook. Monthly fashion spending remains the dependent variable.

## 31. Model 1 — Basic behaviour

Predictors: purchase frequency and discount influence.

In [126]:
model_m1 = ols(
    'Fashion_Spending ~ Purchase_Frequency + Discount_Influence',
    data=analysis_df
).fit()

print(model_m1.summary())

                            OLS Regression Results                            
Dep. Variable:       Fashion_Spending   R-squared:                       0.052
Model:                            OLS   Adj. R-squared:                  0.023
Method:                 Least Squares   F-statistic:                     1.798
Date:                Sun, 23 Aug 2026   Prob (F-statistic):              0.174
Time:                        01:43:13   Log-Likelihood:                -407.20
No. Observations:                  68   AIC:                             820.4
Df Residuals:                      65   BIC:                             827.1
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept            123.2525     43

## 32. Model 2 — Add brand reputation

In [127]:
model_m2 = ols(
    'Fashion_Spending ~ Purchase_Frequency + Discount_Influence + Brand_Reputation_Score',
    data=analysis_df
).fit()

print(model_m2.summary())

                            OLS Regression Results                            
Dep. Variable:       Fashion_Spending   R-squared:                       0.062
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     1.416
Date:                Sun, 23 Aug 2026   Prob (F-statistic):              0.246
Time:                        01:43:13   Log-Likelihood:                -406.85
No. Observations:                  68   AIC:                             821.7
Df Residuals:                      64   BIC:                             830.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 76

## 33. Model 3 — Add social-media influence

In [128]:
model_m3 = ols(
    'Fashion_Spending ~ Purchase_Frequency + Discount_Influence + Brand_Reputation_Score + Social_Media_Influence',
    data=analysis_df
).fit()

print(model_m3.summary())

                            OLS Regression Results                            
Dep. Variable:       Fashion_Spending   R-squared:                       0.065
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     1.091
Date:                Sun, 23 Aug 2026   Prob (F-statistic):              0.369
Time:                        01:43:13   Log-Likelihood:                -406.76
No. Observations:                  68   AIC:                             823.5
Df Residuals:                      63   BIC:                             834.6
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 72

## 34. Model 4 — Full dissertation model

The final model includes income, purchase frequency, discount influence, social-media influence, personalisation and brand reputation.

In [129]:
model_m4 = ols(
    'Fashion_Spending ~ Income_Numeric + Purchase_Frequency + Discount_Influence + Social_Media_Influence + Personalisation_Score + Brand_Reputation_Score',
    data=analysis_df
).fit()

print(model_m4.summary())

                            OLS Regression Results                            
Dep. Variable:       Fashion_Spending   R-squared:                       0.154
Model:                            OLS   Adj. R-squared:                  0.071
Method:                 Least Squares   F-statistic:                     1.855
Date:                Sun, 23 Aug 2026   Prob (F-statistic):              0.103
Time:                        01:43:13   Log-Likelihood:                -403.34
No. Observations:                  68   AIC:                             820.7
Df Residuals:                      61   BIC:                             836.2
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 25

## 35. Model comparison

R² measures explained variance. Adjusted R² accounts for model complexity. AIC balances model fit and complexity, with lower values generally preferred when models use the same sample and dependent variable.

In [130]:
models = {
    'Model 1: Behaviour': model_m1,
    'Model 2: + Brand Reputation': model_m2,
    'Model 3: + Social Media': model_m3,
    'Model 4: Full Model': model_m4
}

comparison = pd.DataFrame({
    'Model': list(models.keys()),
    'R_squared': [m.rsquared for m in models.values()],
    'Adjusted_R_squared': [m.rsquared_adj for m in models.values()],
    'AIC': [m.aic for m in models.values()],
    'BIC': [m.bic for m in models.values()],
    'Model_p_value': [m.f_pvalue for m in models.values()]
})

display(comparison.round(4))

,Model,R_squared,Adjusted_R_squared,AIC,BIC,Model_p_value
0,Model 1: Behaviour,0.0524,0.0233,820.4098,827.0684,0.1737
1,Model 2: + Brand Reputation,0.0622,0.0183,821.7023,830.5804,0.2463
2,Model 3: + Social Media,0.0648,0.0054,823.5167,834.6143,0.3686
3,Model 4: Full Model,0.1543,0.0711,820.6749,836.2115,0.1033


## 36. Interactive model comparison

In [131]:
fig = px.bar(
    comparison,
    x='Model',
    y=['R_squared','Adjusted_R_squared'],
    barmode='group',
    title='Comparison of Multiple Regression Models',
    labels={'value':'Model Fit','variable':'Metric'}
)
fig.show()

# Part F — Full-model diagnostics

The final model is the main inferential model, so its residuals, normality and multicollinearity are checked before interpreting the coefficients.

## 37. Full-model residual plot

In [132]:
full_resid = pd.DataFrame({
    'Fitted': model_m4.fittedvalues,
    'Residual': model_m4.resid
})

fig = px.scatter(
    full_resid,
    x='Fitted',
    y='Residual',
    trendline='lowess',
    title='Full Model: Residuals vs Fitted Values',
    labels={'Fitted':'Fitted Fashion Spending (£)','Residual':'Residual'}
)
fig.add_hline(y=0, line_dash='dash')
fig.show()

## 38. Full-model residual histogram

In [133]:
fig = px.histogram(
    x=model_m4.resid,
    nbins=12,
    marginal='box',
    title='Full Multiple Regression Residual Distribution',
    labels={'x':'Residual'}
)
fig.show()

## 39. Full-model Q-Q plot

In [134]:
theoretical, ordered = stats.probplot(model_m4.resid, dist='norm')[0]
slope, intercept = stats.probplot(model_m4.resid, dist='norm')[1][:2]

qq = pd.DataFrame({
    'Theoretical': theoretical,
    'Observed': ordered
})

xline = np.array([qq['Theoretical'].min(), qq['Theoretical'].max()])
yline = intercept + slope * xline

fig = px.scatter(
    qq,
    x='Theoretical',
    y='Observed',
    title='Q-Q Plot: Full Multiple Regression Residuals',
    labels={'Theoretical':'Theoretical Quantiles','Observed':'Observed Residual Quantiles'}
)

fig.add_trace(
    go.Scatter(
        x=xline,
        y=yline,
        mode='lines',
        name='Reference line'
    )
)

fig.show()

## 40. Full-model Shapiro-Wilk and Durbin-Watson

In [135]:
shapiro_full = stats.shapiro(model_m4.resid)
dw_full = durbin_watson(model_m4.resid)

print(f"Shapiro-Wilk statistic: {shapiro_full.statistic:.4f}")
print(f"Shapiro-Wilk p-value: {shapiro_full.pvalue:.4f}")
print(f"Durbin-Watson statistic: {dw_full:.3f}")

Shapiro-Wilk statistic: 0.9283
Shapiro-Wilk p-value: 0.0008
Durbin-Watson statistic: 2.476


## 41. Multicollinearity — Variance Inflation Factor

VIF is included as an additional diagnostic. Values near 1 indicate very low multicollinearity; substantially larger values require investigation.

In [136]:
vif_vars = [
    'Income_Numeric',
    'Purchase_Frequency',
    'Discount_Influence',
    'Social_Media_Influence',
    'Personalisation_Score',
    'Brand_Reputation_Score'
]

X_vif = sm.add_constant(analysis_df[vif_vars])

vif_table = pd.DataFrame({
    'Variable': X_vif.columns,
    'VIF': [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
})

display(vif_table.round(3))

,Variable,VIF
0,const,39.926
1,Income_Numeric,1.078
2,Purchase_Frequency,1.142
3,Discount_Influence,1.267
4,Social_Media_Influence,1.445
5,Personalisation_Score,1.784
6,Brand_Reputation_Score,1.821


# Part G — Hypotheses and final interpretation

The final model is used to assess the proposed relationships while controlling for the other predictors.

## 42. Full-model coefficient table

In [137]:
coef_table = pd.DataFrame({
    'Coefficient': model_m4.params,
    'Std_Error': model_m4.bse,
    't_value': model_m4.tvalues,
    'p_value': model_m4.pvalues,
    'CI_Lower': model_m4.conf_int()[0],
    'CI_Upper': model_m4.conf_int()[1]
})

display(coef_table.round(4))

,Coefficient,Std_Error,t_value,p_value,CI_Lower,CI_Upper
Intercept,25.3067,73.7415,0.3432,0.7326,-122.1486,172.7619
Income_Numeric,0.0294,0.0117,2.5259,0.0142,0.0061,0.0527
Purchase_Frequency,15.1681,8.9709,1.6908,0.0960,-2.7702,33.1065
Discount_Influence,-13.9760,11.2415,-1.2433,0.2185,-36.4548,8.5027
Social_Media_Influence,12.3106,11.9927,1.0265,0.3087,-11.6703,36.2914
Personalisation_Score,-2.9044,17.0771,-0.1701,0.8655,-37.0522,31.2434
Brand_Reputation_Score,7.8039,15.2059,0.5132,0.6097,-22.6021,38.2099


## 43. Coefficient visualisation

In [138]:
coef_plot = coef_table.drop(index='Intercept').reset_index()
coef_plot.columns = [
    'Variable','Coefficient','Std_Error',
    't_value','p_value','CI_Lower','CI_Upper'
]

fig = px.bar(
    coef_plot,
    x='Coefficient',
    y='Variable',
    orientation='h',
    hover_data=['p_value','CI_Lower','CI_Upper'],
    title='Full Multiple Regression Coefficients',
    labels={'Coefficient':'Estimated Coefficient','Variable':'Predictor'}
)
fig.show()

## 44. Hypothesis testing summary

At the 5% significance level:

- p < 0.05 → statistically significant association
- p ≥ 0.05 → insufficient evidence of a statistically significant association

The result should be reported as an association, not a causal effect.

In [139]:
hypotheses = {
    'H1: Income → Fashion Spending': 'Income_Numeric',
    'H2: Purchase Frequency → Fashion Spending': 'Purchase_Frequency',
    'H3: Discount Influence → Fashion Spending': 'Discount_Influence',
    'H4: Social Media Influence → Fashion Spending': 'Social_Media_Influence',
    'H5: Personalisation → Fashion Spending': 'Personalisation_Score',
    'H6: Brand Reputation → Fashion Spending': 'Brand_Reputation_Score'
}

rows = []

for hypothesis, variable in hypotheses.items():
    p = model_m4.pvalues[variable]
    coef = model_m4.params[variable]

    rows.append({
        'Hypothesis': hypothesis,
        'Coefficient': coef,
        'P-value': p,
        'Decision': (
            'Supported at 5%'
            if p < 0.05
            else 'Not supported at 5%'
        )
    })

hypothesis_table = pd.DataFrame(rows)
display(hypothesis_table.round(4))

,Hypothesis,Coefficient,P-value,Decision
0,H1: Income → Fashion Spending,0.0294,0.0142,Supported at 5%
1,H2: Purchase Frequency → Fashion Spending,15.1681,0.0960,Not supported at 5%
2,H3: Discount Influence → Fashion Spending,-13.9760,0.2185,Not supported at 5%
3,H4: Social Media Influence → Fashion Spending,12.3106,0.3087,Not supported at 5%
4,H5: Personalisation → Fashion Spending,-2.9044,0.8655,Not supported at 5%
5,H6: Brand Reputation → Fashion Spending,7.8039,0.6097,Not supported at 5%


## 45. Final mode

These are the headline statistics to use when writing the dissertation results section.

In [140]:
print("FINAL MODEL SUMMARY")
print("-------------------")
print(f"Observations: {int(model_m4.nobs)}")
print(f"R-squared: {model_m4.rsquared:.4f}")
print(f"Adjusted R-squared: {model_m4.rsquared_adj:.4f}")
print(f"F-statistic: {model_m4.fvalue:.4f}")
print(f"Model p-value: {model_m4.f_pvalue:.6f}")
print(f"AIC: {model_m4.aic:.2f}")
print(f"BIC: {model_m4.bic:.2f}")
print(f"Durbin-Watson: {dw_full:.3f}")
print(f"Shapiro-Wilk p-value: {shapiro_full.pvalue:.4f}")

FINAL MODEL SUMMARY
-------------------
Observations: 68
R-squared: 0.1543
Adjusted R-squared: 0.0711
F-statistic: 1.8552
Model p-value: 0.103278
AIC: 820.67
BIC: 836.21
Durbin-Watson: 2.476
Shapiro-Wilk p-value: 0.0008


In [141]:
significant = coef_table.drop(index='Intercept')
significant = significant[significant['p_value'] < 0.05]

print(
    f"The final model explains {model_m4.rsquared*100:.1f}% "
    f"of the variation in estimated monthly fashion spending."
)

if significant.empty:
    print("No individual predictor is statistically significant at the 5% level in the final model.")
else:
    print("Statistically significant predictors at the 5% level:")
    for variable, row in significant.iterrows():
        direction = "positive" if row['Coefficient'] > 0 else "negative"
        print(
            f"- {variable}: {direction} association; "
            f"coefficient={row['Coefficient']:.3f}, "
            f"p={row['p_value']:.4f}"
        )

print(
    "\nThese results should be interpreted as associations rather than causal effects."
)

The final model explains 15.4% of the variation in estimated monthly fashion spending.
Statistically significant predictors at the 5% level:
- Income_Numeric: positive association; coefficient=0.029, p=0.0142

These results should be interpreted as associations rather than causal effects.


# Final conclusion

This notebook provides a complete quantitative analysis for the dissertation **Customer Behaviour & Personalisation in Fashion Retail**.

The analysis examined the factors influencing fashion spending among 68 consenting respondents. The study investigated income, purchase frequency, discount influence, social-media influence, personalisation and brand reputation to understand their relationship with estimated monthly fashion spending.
The most important finding from the final multiple regression model is that income was the only statistically significant predictor of fashion spending. Income had a positive coefficient of 0.0294 (p = 0.014), indicating that respondents with higher income levels tended to report higher fashion spending. This relationship remained significant after controlling for purchase frequency, discount influence, social-media influence, personalisation and brand reputation. Therefore, income appears to be an important factor when identifying differences in customer spending behaviour.
The other predictors were not statistically significant at the 5% level. Purchase frequency showed a positive relationship with spending (β = 15.17, p = 0.096), but the evidence was insufficient to classify it as statistically significant. Discount influence had a negative coefficient (β = -13.98, p = 0.219), while social-media influence had a positive coefficient (β = 12.31, p = 0.309). Neither relationship was statistically significant. Similarly, personalisation (β = -2.90, p = 0.866) and brand reputation (β = 7.80, p = 0.610) did not significantly predict fashion spending in the final model.
The overall regression model produced an R² of 0.154, meaning that approximately 15.4% of the variation in fashion spending was explained by the six predictors. However, the adjusted R² was only 0.071, and the overall model was not statistically significant (p = 0.103). This indicates that the selected variables provide only limited explanatory power for overall fashion spending. Therefore, fashion retailers should not rely on these variables alone when predicting customer expenditure.
Model comparison also showed that adding more variables did not consistently improve model performance. The full model had a higher R² than the earlier behavioural models, but its adjusted R² remained relatively low. This suggests that some of the additional variables may add complexity without providing sufficient additional explanatory power.
The diagnostic analysis also identified an important limitation. The Shapiro-Wilk test was significant (p = 0.0008), indicating that the residuals do not fully satisfy the normality assumption. Therefore, the regression findings should be interpreted cautiously, particularly given the relatively small sample size. However, the VIF values for the explanatory variables were low, ranging from approximately 1.08 to 1.82, indicating that multicollinearity was not a major concern.
Overall, the findings suggest that income is the strongest statistically supported factor associated with fashion spending in this sample. However, personalisation and brand reputation should not be concluded to have no business value simply because they were not significant predictors of spending. Their influence may operate through other outcomes such as purchase intention, customer experience or loyalty rather than directly determining spending. The results therefore support a more cautious and customer-focused approach to personalisation.

